# Tratativa dos gols
## Etapa tratativa dos dados e gravação na silver

Esta etapa consiste em tratar os dados dos gols, renomear colunas, alterar datatypes e selecionar dados relevantes para o processo.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
# from ast import literal_eval #usar como fallback caso a conversão de string para json não funcione

In [0]:
df_silver = spark.read.table("apifootball.silver.goals") #não existe na primeira execução
max_dh_bronze = (
    spark.read.table("apifootball.bronze.matches_raw")
    .agg(max("dh_processing_br").alias("max_dh"))
    .collect()[0]["max_dh"]
)
df_bronze = (
    spark.read.table("apifootball.bronze.matches_raw").alias("r")
    .filter(col("r.match_status") == "Finished")
    .filter(col("r.dh_processing_br") == max_dh_bronze)
    .join(df_silver.alias("s"), col("r.match_id") == col("s.match_id"), "leftanti") # segunda rodada +, primeira comentar
)#167

In [0]:
df_goals_exploded = (
    df_bronze
    .select(
        "match_id",
        "match_hometeam_id",
        "match_hometeam_name",
        "match_awayteam_id",
        "match_awayteam_name",
        explode("goalscorer").alias("goal"),
        "dh_processing_br"        
    )
)

In [0]:
# Tabela final
df_goals = (
    df_goals_exploded
    .select(
        col("match_id").cast("int").alias("match_id"),
        when(
            col("goal.home_scorer") != "",
            col("goal.home_scorer")
        ).otherwise(
            col("goal.away_scorer")
        ).alias("player_name"),
        when(
            col("goal.home_scorer") != "",
            col("match_hometeam_id")
        ).otherwise(
            col("match_awayteam_id")
        ).cast("int").alias("team_id"),        
        when(
            col("goal.home_scorer") != "",
            col("match_hometeam_name")
        ).otherwise(
            col("match_awayteam_name")
        ).alias("team_name"),
        col("goal.time").alias("goal_time"),        
        col("goal.score").alias("score_after_goal"),
        when(
            col("goal.home_scorer") != "",
            lit(1)
        ).otherwise(
            lit(0)
        ).alias("is_hometeam"),
        when(
            col("goal.away_scorer") != "",
            lit(1)
        ).otherwise(
            lit(0)
        ).alias("is_awayteam"),
        expr('current_timestamp() - INTERVAL 3 HOURS').alias('datetime_processing_br'),
        col('dh_processing_br').alias('datetime_processing_bronze_br')
    )
).orderBy("match_id")

In [0]:
df_goals.write.mode("append").saveAsTable("apifootball.silver.goals")

In [0]:
#spark.read.table("apifootball.silver.goals").display()